In [1]:
import paho.mqtt.client as mqtt
import json
from load_model import load_best_model
from digital_twin import DigitalTwin

# MQTT settings
broker = "mqtt.eclipse.org"
port = 1883
topic = "health_data/update"

# Load the model
model = load_best_model("best_entropy_model.pkl")

# Digital twin instance (initializing with dummy data)
initial_data = {
    "age": 45,
    "trestbps": 120,
    "chol": 210,
    "thalach": 150,
    "oldpeak": 1.5
}
digital_twin = DigitalTwin("Patient_001", initial_data)

# MQTT Callback functions
def on_connect(client, userdata, flags, rc):
    print(f"Connected with result code {rc}")
    client.subscribe(topic)

def on_message(client, userdata, msg):
    print(f"Received message: {msg.payload.decode()}")
    # Parse the received JSON data
    new_data = json.loads(msg.payload.decode())
    digital_twin.update_data(new_data)

    # Get current state of the digital twin and make a prediction
    current_state = list(digital_twin.get_current_state().values())
    prediction = model.predict([current_state])
    prediction_result = "Positive" if prediction[0] == 1 else "Negative"

    print(f"Prediction based on updated data: {prediction_result}")
    # You can send the prediction back or store it

# Set up the MQTT client
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message

# Connect to the broker
client.connect(broker, port, 60)

# Loop to keep the client running
client.loop_start()


ModuleNotFoundError: No module named 'load_model'